In [1]:
import pandas as pd
import os

seasons = [
    "18-19", "19-20", "20-21",
    "21-22", "22-23", "23-24",
    "24-25", "25-26", "26-27"
]

dfs = []

for division in ["D1", "D2"]:
    for season in seasons:
        file = f"rawdata/{division} {season}.csv"

        temp = pd.read_csv(file)

        temp["Season"] = season
        temp["Division"] = division

        dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Dataset shape:", df.shape)
print(df[["Season", "Division", "HomeTeam", "AwayTeam", "FTR"]].head())

Dataset shape: (5076, 195)
  Season Division            HomeTeam       AwayTeam FTR
0  18-19       D1       Bayern Munich     Hoffenheim   H
1  18-19       D1  Fortuna Dusseldorf       Augsburg   A
2  18-19       D1            Freiburg  Ein Frankfurt   A
3  18-19       D1              Hertha       Nurnberg   H
4  18-19       D1          M'gladbach     Leverkusen   H


In [2]:
df_d1 = df[df["Division"] == "D1"].copy()
df_d2 = df[df["Division"] == "D2"].copy()

df_d1.reset_index(drop=True, inplace=True)
df_d2.reset_index(drop=True, inplace=True)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

Bundesliga 1: (2529, 195)
Bundesliga 2: (2547, 195)


In [3]:
# Correct missing match statistics for Union Berlin vs Bochum (December 14, 2024).
# The match ended 1-1 on the field but was officially awarded 0-2 to Bochum.
# Keep the official result while restoring actual match statistics.

match = (
    (df_d1["Season"] == "24-25") &
    (df_d1["HomeTeam"] == "Union Berlin") &
    (df_d1["AwayTeam"] == "Bochum")
)

corrections = {
    "HTHG": 1, "HTAG": 1, "HTR": "D",
    "HS": 22, "AS": 13,
    "HST": 4, "AST": 3,
    "HC": 7, "AC": 6,
    "HF": 11, "AF": 9,
    "HY": 2, "AY": 2,
    "HR": 0, "AR": 1
}

assert match.sum() == 1, "Expected exactly one matching game"

for col, value in corrections.items():
    df_d1.loc[match, col] = value

print("Corrected match:")
display(df_d1.loc[match])

Corrected match:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,PPA,SKBH,SKBD,SKBA,PPCH,PPCD,PPCA,SKBCH,SKBCD,SKBCA
1957,D1,14/12/2024,Union Berlin,Bochum,0,2,A,1.0,1.0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
keep_cols = [
    "Season", "Division", "Date",
    "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR",
    "HTHG", "HTAG", "HTR",
    "HS", "AS", "HST", "AST",
    "HC", "AC", "HF", "AF",
    "HY", "AY", "HR", "AR"
]

df_d1 = df_d1[keep_cols].copy()
df_d2 = df_d2[keep_cols].copy()

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

print("\nMissing values in Bundesliga 1:")
print(df_d1.isnull().sum())

print("\nMissing values in Bundesliga 2:")
print(df_d2.isnull().sum())

Bundesliga 1: (2529, 23)
Bundesliga 2: (2547, 23)

Missing values in Bundesliga 1:
Season      0
Division    0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
dtype: int64

Missing values in Bundesliga 2:
Season      0
Division    0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
dtype: int64


In [5]:
# Identify Bundesliga 1 matches with missing statistics

missing_matches = df_d1[df_d1.isnull().any(axis=1)]

print(missing_matches.to_string(index=False))

Empty DataFrame
Columns: [Season, Division, Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR, HTHG, HTAG, HTR, HS, AS, HST, AST, HC, AC, HF, AF, HY, AY, HR, AR]
Index: []


In [6]:
# Convert match dates to datetime and sort both divisions chronologically

for data in [df_d1, df_d2]:
    data["Date"] = pd.to_datetime(
        data["Date"],
        format="%d/%m/%Y"
    )

    data.sort_values(
        by=["Date", "Season"],
        inplace=True
    )

    data.reset_index(drop=True, inplace=True)

print("Bundesliga 1:")
print(df_d1[["Date", "HomeTeam", "AwayTeam"]].head())

print("\nBundesliga 2:")
print(df_d2[["Date", "HomeTeam", "AwayTeam"]].head())

Bundesliga 1:
        Date            HomeTeam       AwayTeam
0 2018-08-24       Bayern Munich     Hoffenheim
1 2018-08-25  Fortuna Dusseldorf       Augsburg
2 2018-08-25            Freiburg  Ein Frankfurt
3 2018-08-25              Hertha       Nurnberg
4 2018-08-25          M'gladbach     Leverkusen

Bundesliga 2:
        Date        HomeTeam       AwayTeam
0 2018-08-03         Hamburg  Holstein Kiel
1 2018-08-04          Bochum        FC Koln
2 2018-08-04  Greuther Furth     Sandhausen
3 2018-08-04      Regensburg     Ingolstadt
4 2018-08-05       Darmstadt      Paderborn


In [7]:
# Check unique team names across both divisions and all seasons

teams_d1 = sorted(
    set(df_d1["HomeTeam"]) | set(df_d1["AwayTeam"])
)

teams_d2 = sorted(
    set(df_d2["HomeTeam"]) | set(df_d2["AwayTeam"])
)

print("Bundesliga 1 teams:", len(teams_d1))
print(teams_d1)

print("\nBundesliga 2 teams:", len(teams_d2))
print(teams_d2)

Bundesliga 1 teams: 30
['Augsburg', 'Bayern Munich', 'Bielefeld', 'Bochum', 'Darmstadt', 'Dortmund', 'Ein Frankfurt', 'Elversberg', 'FC Koln', 'Fortuna Dusseldorf', 'Freiburg', 'Greuther Furth', 'Hamburg', 'Hannover', 'Heidenheim', 'Hertha', 'Hoffenheim', 'Holstein Kiel', 'Leverkusen', "M'gladbach", 'Mainz', 'Nurnberg', 'Paderborn', 'RB Leipzig', 'Schalke 04', 'St Pauli', 'Stuttgart', 'Union Berlin', 'Werder Bremen', 'Wolfsburg']

Bundesliga 2 teams: 37
['Bielefeld', 'Bochum', 'Braunschweig', 'Cottbus', 'Darmstadt', 'Dresden', 'Duisburg', 'Elversberg', 'Erzgebirge Aue', 'FC Koln', 'Fortuna Dusseldorf', 'Greuther Furth', 'Hamburg', 'Hannover', 'Hansa Rostock', 'Heidenheim', 'Hertha', 'Holstein Kiel', 'Ingolstadt', 'Kaiserslautern', 'Karlsruhe', 'Magdeburg', 'Nurnberg', 'Osnabruck', 'Paderborn', 'Preußen Münster', 'Regensburg', 'Sandhausen', 'Schalke 04', 'St Pauli', 'Stuttgart', 'Ulm', 'Union Berlin', 'Wehen', 'Werder Bremen', 'Wolfsburg', 'Wurzburger Kickers']


In [8]:
# Check for duplicate matches using season, date, and team names

match_cols = ["Season", "Date", "HomeTeam", "AwayTeam"]

duplicates_d1 = df_d1[df_d1.duplicated(subset=match_cols, keep=False)]
duplicates_d2 = df_d2[df_d2.duplicated(subset=match_cols, keep=False)]

print("Bundesliga 1 duplicate matches:", len(duplicates_d1))
print("Bundesliga 2 duplicate matches:", len(duplicates_d2))

display(duplicates_d1)
display(duplicates_d2)

Bundesliga 1 duplicate matches: 0
Bundesliga 2 duplicate matches: 0


,Season,Division,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR


,Season,Division,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR


In [9]:
# Calculate points earned by home and away teams in each match

points_map = {
    "H": (3, 0),
    "D": (1, 1),
    "A": (0, 3)
}

for data in [df_d1, df_d2]:
    data["HomePoints"] = data["FTR"].map(lambda x: points_map[x][0])
    data["AwayPoints"] = data["FTR"].map(lambda x: points_map[x][1])

print("Bundesliga 1:")
display(df_d1[["HomeTeam", "AwayTeam", "FTR", "HomePoints", "AwayPoints"]].head())

print("Bundesliga 2:")
display(df_d2[["HomeTeam", "AwayTeam", "FTR", "HomePoints", "AwayPoints"]].head())

Bundesliga 1:


,HomeTeam,AwayTeam,FTR,HomePoints,AwayPoints
0,Bayern Munich,Hoffenheim,H,3,0
1,Fortuna Dusseldorf,Augsburg,A,0,3
2,Freiburg,Ein Frankfurt,A,0,3
3,Hertha,Nurnberg,H,3,0
4,M'gladbach,Leverkusen,H,3,0


Bundesliga 2:


,HomeTeam,AwayTeam,FTR,HomePoints,AwayPoints
0,Hamburg,Holstein Kiel,A,0,3
1,Bochum,FC Koln,A,0,3
2,Greuther Furth,Sandhausen,H,3,0
3,Regensburg,Ingolstadt,H,3,0
4,Darmstadt,Paderborn,H,3,0


In [10]:
# Create a team-level match history for both Bundesliga divisions.
# Previously, each row represented one match with two teams (home and away).
# This function separates each match into two rows, one from each team's perspective.
# It records each team's points, goals, shots, corners, fouls, and cards,
# including both their own statistics ("For") and their opponent's ("Against").
# Finally, it combines the home/away records and sorts them by team and date.


def create_team_history(data):
    stat_cols = {
        "Goals": ("FTHG", "FTAG"),
        "Shots": ("HS", "AS"),
        "ShotsOnTarget": ("HST", "AST"),
        "Corners": ("HC", "AC"),
        "Fouls": ("HF", "AF"),
        "YellowCards": ("HY", "AY"),
        "RedCards": ("HR", "AR")
    }

    histories = []

    for venue in ["Home", "Away"]:
        temp = data[[
            "Season", "Division", "Date",
            "HomeTeam", "AwayTeam", "HomePoints", "AwayPoints"
        ] + [col for pair in stat_cols.values() for col in pair]].copy()

        temp["Team"] = temp["HomeTeam"] if venue == "Home" else temp["AwayTeam"]
        temp["Opponent"] = temp["AwayTeam"] if venue == "Home" else temp["HomeTeam"]
        temp["Points"] = temp["HomePoints"] if venue == "Home" else temp["AwayPoints"]
        temp["Venue"] = venue

        for stat, (home_col, away_col) in stat_cols.items():
            temp[f"{stat}For"] = temp[home_col] if venue == "Home" else temp[away_col]
            temp[f"{stat}Against"] = temp[away_col] if venue == "Home" else temp[home_col]

        keep = ["Season", "Division", "Date", "Team", "Opponent", "Points", "Venue"]
        keep += [f"{stat}{side}" for stat in stat_cols for side in ["For", "Against"]]

        histories.append(temp[keep])

    history = pd.concat(histories, ignore_index=True)
    history = history.sort_values(["Team", "Date"]).reset_index(drop=True)

    return history


history_d1 = create_team_history(df_d1)
history_d2 = create_team_history(df_d2)

print("Bundesliga 1:", history_d1.shape)
print("Bundesliga 2:", history_d2.shape)

display(history_d1.head())

Bundesliga 1: (5058, 21)
Bundesliga 2: (5094, 21)


,Season,Division,Date,Team,Opponent,Points,Venue,GoalsFor,GoalsAgainst,ShotsFor,...,ShotsOnTargetFor,ShotsOnTargetAgainst,CornersFor,CornersAgainst,FoulsFor,FoulsAgainst,YellowCardsFor,YellowCardsAgainst,RedCardsFor,RedCardsAgainst
0,18-19,D1,2018-08-25,Augsburg,Fortuna Dusseldorf,3,Away,2,1,14.0,...,7.0,5.0,5.0,7.0,10.0,11.0,0.0,1.0,0.0,0.0
1,18-19,D1,2018-09-01,Augsburg,M'gladbach,1,Home,1,1,18.0,...,7.0,2.0,9.0,4.0,12.0,12.0,1.0,2.0,0.0,0.0
2,18-19,D1,2018-09-15,Augsburg,Mainz,0,Away,1,2,13.0,...,4.0,7.0,4.0,5.0,16.0,9.0,2.0,1.0,0.0,0.0
3,18-19,D1,2018-09-22,Augsburg,Werder Bremen,0,Home,2,3,21.0,...,7.0,7.0,7.0,4.0,10.0,11.0,2.0,2.0,0.0,0.0
4,18-19,D1,2018-09-25,Augsburg,Bayern Munich,1,Away,1,1,12.0,...,4.0,8.0,4.0,8.0,15.0,6.0,2.0,3.0,0.0,0.0


In [11]:
# Calculate each team's statistics from their previous 5 matches.
# Group by team and season so rolling statistics reset each season.
# Use shift(1) to exclude the current match and prevent data leakage.
# Require 5 previous matches before calculating any rolling features.

def add_rolling_features(history):
    history = history.copy()

    stat_cols = [
        "Points",
        "GoalsFor", "GoalsAgainst",
        "ShotsFor", "ShotsAgainst",
        "ShotsOnTargetFor", "ShotsOnTargetAgainst",
        "CornersFor", "CornersAgainst",
        "FoulsFor", "FoulsAgainst",
        "YellowCardsFor", "YellowCardsAgainst",
        "RedCardsFor", "RedCardsAgainst"
    ]

    for col in stat_cols:
        grouped = history.groupby(["Season", "Team"])[col]

        if col == "Points":
            history["Last5Points"] = grouped.transform(
                lambda x: x.shift(1).rolling(5, min_periods=5).sum()
            )
        else:
            history[f"Last5Avg{col}"] = grouped.transform(
                lambda x: x.shift(1).rolling(5, min_periods=5).mean()
            )

    return history


history_d1 = add_rolling_features(history_d1)
history_d2 = add_rolling_features(history_d2)

display(
    history_d1[
        ["Season", "Date", "Team", "Points",
         "Last5Points", "Last5AvgGoalsFor", "Last5AvgGoalsAgainst"]
    ].head(10)
)

,Season,Date,Team,Points,Last5Points,Last5AvgGoalsFor,Last5AvgGoalsAgainst
0,18-19,2018-08-25,Augsburg,3,NaN,NaN,NaN
1,18-19,2018-09-01,Augsburg,1,NaN,NaN,NaN
2,18-19,2018-09-15,Augsburg,0,NaN,NaN,NaN
3,18-19,2018-09-22,Augsburg,0,NaN,NaN,NaN
4,18-19,2018-09-25,Augsburg,1,NaN,NaN,NaN
5,18-19,2018-09-30,Augsburg,3,5.0,1.4,1.6
6,18-19,2018-10-06,Augsburg,0,5.0,1.8,1.6
7,18-19,2018-10-20,Augsburg,1,4.0,2.2,2.2
8,18-19,2018-10-27,Augsburg,3,5.0,2.0,1.8
9,18-19,2018-11-03,Augsburg,1,8.0,2.0,1.4


In [12]:
# Merge each team's previous 5-game statistics into the original match data.
# Create separate Home_ and Away_ features for each match.
# Match using season, date, and team names to attach the correct history.

def merge_rolling_features(data, history):
    rolling_cols = [
        col for col in history.columns
        if col.startswith("Last5")
    ]

    home = history[history["Venue"] == "Home"][
        ["Season", "Date", "Team", "Opponent"] + rolling_cols
    ].copy()

    away = history[history["Venue"] == "Away"][
        ["Season", "Date", "Team", "Opponent"] + rolling_cols
    ].copy()

    home = home.rename(columns={
        "Team": "HomeTeam",
        "Opponent": "AwayTeam",
        **{col: f"Home_{col}" for col in rolling_cols}
    })

    away = away.rename(columns={
        "Team": "AwayTeam",
        "Opponent": "HomeTeam",
        **{col: f"Away_{col}" for col in rolling_cols}
    })

    match_keys = ["Season", "Date", "HomeTeam", "AwayTeam"]

    data = data.merge(home, on=match_keys, how="left", validate="one_to_one")
    data = data.merge(away, on=match_keys, how="left", validate="one_to_one")

    return data


df_d1 = merge_rolling_features(df_d1, history_d1)
df_d2 = merge_rolling_features(df_d2, history_d2)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam",
     "Home_Last5Points", "Away_Last5Points", "FTR"]
].head(15))

Bundesliga 1: (2529, 55)
Bundesliga 2: (2547, 55)


,Date,HomeTeam,AwayTeam,Home_Last5Points,Away_Last5Points,FTR
0,2018-08-24,Bayern Munich,Hoffenheim,NaN,NaN,H
1,2018-08-25,Fortuna Dusseldorf,Augsburg,NaN,NaN,A
2,2018-08-25,Freiburg,Ein Frankfurt,NaN,NaN,A
3,2018-08-25,Hertha,Nurnberg,NaN,NaN,H
4,2018-08-25,M'gladbach,Leverkusen,NaN,NaN,H
5,2018-08-25,Werder Bremen,Hannover,NaN,NaN,D
6,2018-08-25,Wolfsburg,Schalke 04,NaN,NaN,H
7,2018-08-26,Dortmund,RB Leipzig,NaN,NaN,H
8,2018-08-26,Mainz,Stuttgart,NaN,NaN,H
9,2018-08-31,Hannover,Dortmund,NaN,NaN,D


In [13]:
# Verify that rolling features were merged correctly for both divisions.
# Count matches with complete five-game features and inspect sample matches.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:
    valid = data.dropna(
        subset=["Home_Last5Points", "Away_Last5Points"]
    )

    print(f"{name}:")
    print("Matches with both teams having 5 previous games:", len(valid))
    print("Matches without complete 5-game points:", len(data) - len(valid))

    display(valid[
        ["Season", "Date", "HomeTeam", "AwayTeam",
         "Home_Last5Points", "Away_Last5Points", "FTR"]
    ].head())

Bundesliga 1:
Matches with both teams having 5 previous games: 2124
Matches without complete 5-game points: 405


,Season,Date,HomeTeam,AwayTeam,Home_Last5Points,Away_Last5Points,FTR
45,18-19,2018-09-28,Hertha,Bayern Munich,10.0,13.0,H
46,18-19,2018-09-29,Hoffenheim,RB Leipzig,7.0,8.0,A
47,18-19,2018-09-29,Leverkusen,Dortmund,6.0,11.0,A
48,18-19,2018-09-29,Nurnberg,Fortuna Dusseldorf,5.0,5.0,H
49,18-19,2018-09-29,Schalke 04,Mainz,0.0,8.0,H


Bundesliga 2:
Matches with both teams having 5 previous games: 2142
Matches without complete 5-game points: 405


,Season,Date,HomeTeam,AwayTeam,Home_Last5Points,Away_Last5Points,FTR
45,18-19,2018-09-21,Ingolstadt,St Pauli,5.0,6.0,A
46,18-19,2018-09-21,Sandhausen,FC Koln,2.0,10.0,A
47,18-19,2018-09-22,Bielefeld,Union Berlin,8.0,9.0,D
48,18-19,2018-09-22,Dresden,Darmstadt,6.0,10.0,H
49,18-19,2018-09-22,Holstein Kiel,Bochum,8.0,10.0,D


In [14]:
# Calculate total points earned by each team in each season.
# Combine both divisions so we can track teams across promotions and relegations.
# This creates a season-level summary for future historical points features.

all_history = pd.concat(
    [history_d1, history_d2],
    ignore_index=True
)

season_points = (
    all_history
    .groupby(["Season", "Division", "Team"], as_index=False)
    .agg(
        SeasonPoints=("Points", "sum"),
        MatchesPlayed=("Points", "count")
    )
)

season_points["PointsPerGame"] = (
    season_points["SeasonPoints"] / season_points["MatchesPlayed"]
)

display(season_points.head(10))

,Season,Division,Team,SeasonPoints,MatchesPlayed,PointsPerGame
0,18-19,D1,Augsburg,32,34,0.941176
1,18-19,D1,Bayern Munich,78,34,2.294118
2,18-19,D1,Dortmund,76,34,2.235294
3,18-19,D1,Ein Frankfurt,54,34,1.588235
4,18-19,D1,Fortuna Dusseldorf,44,34,1.294118
5,18-19,D1,Freiburg,36,34,1.058824
6,18-19,D1,Hannover,21,34,0.617647
7,18-19,D1,Hertha,43,34,1.264706
8,18-19,D1,Hoffenheim,51,34,1.500000
9,18-19,D1,Leverkusen,58,34,1.705882


In [15]:
# Calculate historical team performance using up to 5 previous seasons.
# Use shift(1) to exclude the current season and prevent data leakage.
# Group by team across both divisions to account for promotion and relegation.
# Track how many previous seasons are available for each team.

season_order = {season: i for i, season in enumerate(seasons)}

season_points = season_points.copy()
season_points["SeasonOrder"] = season_points["Season"].map(season_order)

season_points = season_points.sort_values(
    ["Team", "SeasonOrder"]
).reset_index(drop=True)

grouped = season_points.groupby("Team")

season_points["Prev5SeasonAvgPPG"] = grouped["PointsPerGame"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

season_points["Prev5SeasonAvgPoints"] = grouped["SeasonPoints"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

season_points["PrevSeasonsAvailable"] = grouped["Season"].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).count()
)

display(
    season_points[
        ["Season", "Division", "Team", "SeasonPoints",
         "Prev5SeasonAvgPPG", "Prev5SeasonAvgPoints",
         "PrevSeasonsAvailable"]
    ].query("Team == 'Bayern Munich'")
)

,Season,Division,Team,SeasonPoints,Prev5SeasonAvgPPG,Prev5SeasonAvgPoints,PrevSeasonsAvailable
9,18-19,D1,Bayern Munich,78,NaN,NaN,0.0
10,19-20,D1,Bayern Munich,82,2.294118,78.000000,1.0
11,20-21,D1,Bayern Munich,78,2.352941,80.000000,2.0
12,21-22,D1,Bayern Munich,77,2.333333,79.333333,3.0
13,22-23,D1,Bayern Munich,71,2.316176,78.750000,4.0
14,23-24,D1,Bayern Munich,72,2.270588,77.200000,5.0
15,24-25,D1,Bayern Munich,82,2.235294,76.000000,5.0
16,25-26,D1,Bayern Munich,89,2.235294,76.000000,5.0
17,26-27,D1,Bayern Munich,21,2.300000,78.200000,5.0


In [16]:
# Merge previous 5-season performance into each match.
# Match teams by season and name, accounting for promotion/relegation.
# Add separate Home_ and Away_ historical features for modeling.

def merge_season_features(data, season_points):
    feature_cols = [
        "Prev5SeasonAvgPPG",
        "Prev5SeasonAvgPoints",
        "PrevSeasonsAvailable"
    ]

    home = season_points[
        ["Season", "Team"] + feature_cols
    ].rename(columns={
        "Team": "HomeTeam",
        **{col: f"Home_{col}" for col in feature_cols}
    })

    away = season_points[
        ["Season", "Team"] + feature_cols
    ].rename(columns={
        "Team": "AwayTeam",
        **{col: f"Away_{col}" for col in feature_cols}
    })

    data = data.merge(
        home,
        on=["Season", "HomeTeam"],
        how="left",
        validate="many_to_one"
    )

    data = data.merge(
        away,
        on=["Season", "AwayTeam"],
        how="left",
        validate="many_to_one"
    )

    return data


df_d1 = merge_season_features(df_d1, season_points)
df_d2 = merge_season_features(df_d2, season_points)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Season", "HomeTeam", "AwayTeam",
     "Home_Prev5SeasonAvgPPG", "Away_Prev5SeasonAvgPPG"]
].tail(10))

Bundesliga 1: (2529, 61)
Bundesliga 2: (2547, 61)


,Season,HomeTeam,AwayTeam,Home_Prev5SeasonAvgPPG,Away_Prev5SeasonAvgPPG
2519,26-27,Ein Frankfurt,Hamburg,1.429412,1.652941
2520,26-27,Hamburg,M'gladbach,1.652941,1.205882
2521,26-27,Dortmund,Elversberg,1.958824,1.598039
2522,26-27,Mainz,Bayern Munich,1.288235,2.300000
2523,26-27,Paderborn,Ein Frankfurt,1.617647,1.429412
2524,26-27,RB Leipzig,Augsburg,1.794118,1.158824
2525,26-27,Stuttgart,Werder Bremen,1.476471,1.317647
2526,26-27,FC Koln,Leverkusen,1.258824,1.952941
2527,26-27,Freiburg,Union Berlin,1.517647,1.358824
2528,26-27,Hoffenheim,Schalke 04,1.300000,1.452941


In [17]:
# Calculate head-to-head records from the previous 5 meetings.
# Combine both divisions to include meetings before promotion/relegation.
# Use only earlier match dates to prevent data leakage.
# Count wins for the current home team, wins for the current away team, and draws.

all_matches = pd.concat([df_d1, df_d2], ignore_index=True)
all_matches = all_matches.sort_values("Date").copy()

h2h_records = {}

for idx, match in all_matches.iterrows():
    home = match["HomeTeam"]
    away = match["AwayTeam"]

    # Use a sorted pair so the matchup is the same regardless of venue
    pair = tuple(sorted([home, away]))

    previous = h2h_records.get(pair, [])

    # Only use meetings strictly before the current date
    previous_five = [m for m in previous if m[0] < match["Date"]][-5:]

    all_matches.loc[idx, "H2H_HomeWins"] = sum(
        m[1] == home for m in previous_five
    )

    all_matches.loc[idx, "H2H_AwayWins"] = sum(
        m[1] == away for m in previous_five
    )

    all_matches.loc[idx, "H2H_Draws"] = sum(
        m[1] is None for m in previous_five
    )

    all_matches.loc[idx, "H2H_Matches"] = len(previous_five)

    # Record the winner after calculating historical features
    if match["FTR"] == "H":
        winner = home
    elif match["FTR"] == "A":
        winner = away
    else:
        winner = None

    h2h_records.setdefault(pair, []).append((match["Date"], winner))


df_d1 = all_matches[all_matches["Division"] == "D1"].copy()
df_d2 = all_matches[all_matches["Division"] == "D2"].copy()

df_d1.reset_index(drop=True, inplace=True)
df_d2.reset_index(drop=True, inplace=True)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam",
     "H2H_HomeWins", "H2H_AwayWins", "H2H_Draws", "H2H_Matches"]
].tail(10))

Bundesliga 1: (2529, 65)
Bundesliga 2: (2547, 65)


,Date,HomeTeam,AwayTeam,H2H_HomeWins,H2H_AwayWins,H2H_Draws,H2H_Matches
2519,2026-11-01,Ein Frankfurt,Hamburg,0.0,1.0,1.0,2.0
2520,2026-11-06,Hamburg,M'gladbach,0.0,0.0,2.0,2.0
2521,2026-11-07,Mainz,Bayern Munich,1.0,3.0,1.0,5.0
2522,2026-11-07,Paderborn,Ein Frankfurt,1.0,1.0,0.0,2.0
2523,2026-11-07,RB Leipzig,Augsburg,3.0,0.0,2.0,5.0
2524,2026-11-07,Stuttgart,Werder Bremen,1.0,2.0,2.0,5.0
2525,2026-11-07,Dortmund,Elversberg,0.0,0.0,0.0,0.0
2526,2026-11-08,Hoffenheim,Schalke 04,3.0,1.0,1.0,5.0
2527,2026-11-08,FC Koln,Leverkusen,1.0,4.0,0.0,5.0
2528,2026-11-08,Freiburg,Union Berlin,0.0,3.0,2.0,5.0


In [18]:
# Calculate the number of days since each team's previous match.
# Use both Bundesliga divisions to track teams across promotion/relegation.
# Sort by team and date, then use groupby() and diff() to calculate rest days.
# Merge the results back into the match datasets for home and away teams.

def add_rest_days(df_d1, df_d2):
    all_matches = pd.concat([df_d1, df_d2], ignore_index=True)

    home = all_matches[
        ["Season", "Date", "HomeTeam", "AwayTeam"]
    ].copy()
    home["Team"] = home["HomeTeam"]
    home["Venue"] = "Home"

    away = all_matches[
        ["Season", "Date", "HomeTeam", "AwayTeam"]
    ].copy()
    away["Team"] = away["AwayTeam"]
    away["Venue"] = "Away"

    history = pd.concat([home, away], ignore_index=True)
    history = history.sort_values(["Team", "Date"])

    history["DaysSinceLastMatch"] = (
        history.groupby("Team")["Date"]
        .diff()
        .dt.days
    )

    keys = ["Season", "Date", "HomeTeam", "AwayTeam"]

    home_rest = history[history["Venue"] == "Home"][
        keys + ["DaysSinceLastMatch"]
    ].rename(columns={
        "DaysSinceLastMatch": "Home_RestDays"
    })

    away_rest = history[history["Venue"] == "Away"][
        keys + ["DaysSinceLastMatch"]
    ].rename(columns={
        "DaysSinceLastMatch": "Away_RestDays"
    })

    all_matches = all_matches.merge(
        home_rest, on=keys, how="left", validate="one_to_one"
    )

    all_matches = all_matches.merge(
        away_rest, on=keys, how="left", validate="one_to_one"
    )

    return (
        all_matches[all_matches["Division"] == "D1"].reset_index(drop=True),
        all_matches[all_matches["Division"] == "D2"].reset_index(drop=True)
    )


df_d1, df_d2 = add_rest_days(df_d1, df_d2)

print("Bundesliga 1:", df_d1.shape)
print("Bundesliga 2:", df_d2.shape)

display(df_d1[
    ["Date", "HomeTeam", "AwayTeam", "Home_RestDays", "Away_RestDays"]
].head(15))

Bundesliga 1: (2529, 67)
Bundesliga 2: (2547, 67)


,Date,HomeTeam,AwayTeam,Home_RestDays,Away_RestDays
0,2018-08-24,Bayern Munich,Hoffenheim,NaN,NaN
1,2018-08-25,Fortuna Dusseldorf,Augsburg,NaN,NaN
2,2018-08-25,Wolfsburg,Schalke 04,NaN,NaN
3,2018-08-25,Werder Bremen,Hannover,NaN,NaN
4,2018-08-25,M'gladbach,Leverkusen,NaN,NaN
5,2018-08-25,Hertha,Nurnberg,NaN,NaN
6,2018-08-25,Freiburg,Ein Frankfurt,NaN,NaN
7,2018-08-26,Mainz,Stuttgart,NaN,NaN
8,2018-08-26,Dortmund,RB Leipzig,NaN,NaN
9,2018-08-31,Hannover,Dortmund,6.0,5.0


In [19]:
# Check whether the original datasets contain stadium or attendance information.
# Search all original column names for relevant keywords before using external data.

keywords = ["stadium", "capacity", "attendance", "venue", "crowd"]

stadium_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in keywords)
]

print("Potential stadium-related columns:", stadium_cols)

Potential stadium-related columns: []


In [20]:
# Check missing values in all engineered features for both Bundesliga divisions.
# Identify which features have missing data and how many matches are affected.
# This helps us decide which rows to remove or which values to handle.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:

    feature_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5",
                           "Home_Prev5", "Away_Prev5",
                           "Home_PrevSeasons", "Away_PrevSeasons",
                           "H2H_", "Home_Rest", "Away_Rest"))
    ]

    missing = data[feature_cols].isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f"\n{name}")
    print("Total matches:", len(data))
    print("Features with missing values:")
    print(missing.to_string())

    print("\nMatches with all engineered features available:",
          data[feature_cols].notnull().all(axis=1).sum())


Bundesliga 1
Total matches: 2529
Features with missing values:
Home_Last5Points                     405
Home_Last5AvgGoalsFor                405
Home_Last5AvgGoalsAgainst            405
Home_Last5AvgShotsFor                405
Home_Last5AvgShotsAgainst            405
Home_Last5AvgShotsOnTargetFor        405
Home_Last5AvgShotsOnTargetAgainst    405
Home_Last5AvgCornersFor              405
Home_Last5AvgCornersAgainst          405
Home_Last5AvgFoulsFor                405
Home_Last5AvgFoulsAgainst            405
Home_Last5AvgYellowCardsFor          405
Home_Last5AvgYellowCardsAgainst      405
Home_Last5AvgRedCardsFor             405
Home_Last5AvgRedCardsAgainst         405
Away_Last5Points                     405
Away_Last5AvgGoalsFor                405
Away_Last5AvgGoalsAgainst            405
Away_Last5AvgShotsFor                405
Away_Last5AvgShotsAgainst            405
Away_Last5AvgShotsOnTargetFor        405
Away_Last5AvgShotsOnTargetAgainst    405
Away_Last5AvgCornersFor           

In [21]:
# Check how many matches have complete previous 5-game statistics.
# Do not remove rows yet; only calculate how many would be retained.
# Historical season averages and rest days will be handled separately.

for name, data in [("Bundesliga 1", df_d1), ("Bundesliga 2", df_d2)]:

    rolling_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ]

    complete_rolling = data[rolling_cols].notna().all(axis=1)

    print(f"\n{name}")
    print("Total matches:", len(data))
    print("Matches with complete rolling features:", complete_rolling.sum())
    print("Matches removed:", (~complete_rolling).sum())

    print("\nRemaining missing values:")
    remaining = data.loc[complete_rolling]
    missing = remaining.isna().sum()
    print(missing[missing > 0].to_string())


Bundesliga 1
Total matches: 2529
Matches with complete rolling features: 2124
Matches removed: 405

Remaining missing values:
Home_Prev5SeasonAvgPPG       261
Home_Prev5SeasonAvgPoints    261
Away_Prev5SeasonAvgPPG       261
Away_Prev5SeasonAvgPoints    261

Bundesliga 2
Total matches: 2547
Matches with complete rolling features: 2142
Matches removed: 405

Remaining missing values:
Home_Prev5SeasonAvgPPG       408
Home_Prev5SeasonAvgPoints    408
Away_Prev5SeasonAvgPPG       410
Away_Prev5SeasonAvgPoints    410


In [22]:
# Create final working datasets using matches with complete previous 5-game features.
# Exclude matches without enough historical match statistics.
# Preserve missing previous-season averages for later model preprocessing.
# Keep the original DataFrames unchanged for validation.

rolling_cols_d1 = [
    col for col in df_d1.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

rolling_cols_d2 = [
    col for col in df_d2.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

final_d1 = df_d1.dropna(subset=rolling_cols_d1).copy()
final_d2 = df_d2.dropna(subset=rolling_cols_d2).copy()

final_d1.reset_index(drop=True, inplace=True)
final_d2.reset_index(drop=True, inplace=True)

print("Final Bundesliga 1:", final_d1.shape)
print("Final Bundesliga 2:", final_d2.shape)

Final Bundesliga 1: (2124, 67)
Final Bundesliga 2: (2142, 67)


In [23]:
# Create final working datasets using matches with complete previous 5-game features.
# Exclude matches without enough historical match statistics.
# Preserve missing previous-season averages for later model preprocessing.
# Keep the original DataFrames unchanged for validation.

rolling_cols_d1 = [
    col for col in df_d1.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

rolling_cols_d2 = [
    col for col in df_d2.columns
    if col.startswith(("Home_Last5", "Away_Last5"))
]

final_d1 = df_d1.dropna(subset=rolling_cols_d1).copy()
final_d2 = df_d2.dropna(subset=rolling_cols_d2).copy()

final_d1.reset_index(drop=True, inplace=True)
final_d2.reset_index(drop=True, inplace=True)

print("Final Bundesliga 1:", final_d1.shape)
print("Final Bundesliga 2:", final_d2.shape)

Final Bundesliga 1: (2124, 67)
Final Bundesliga 2: (2142, 67)


In [24]:
# Validate both final datasets before exporting.
# Check duplicates, target values, rolling features, and historical feature ranges.
# Identify any remaining missing values for teammates to handle during modeling.

for name, data in [("Bundesliga 1", final_d1), ("Bundesliga 2", final_d2)]:

    print(f"\n{name}")
    print("Dataset shape:", data.shape)

    # Check duplicate matches
    duplicates = data.duplicated(
        subset=["Season", "Date", "HomeTeam", "AwayTeam"]
    ).sum()
    print("Duplicate matches:", duplicates)

    # Check match outcomes
    print("Match outcomes:", data["FTR"].value_counts().to_dict())
    print("Invalid outcomes:", (~data["FTR"].isin(["H", "D", "A"])).sum())

    # Check rolling features
    rolling_cols = [
        col for col in data.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ]
    print("Missing rolling values:", data[rolling_cols].isna().sum().sum())

    # Check points are between 0 and 15
    for col in ["Home_Last5Points", "Away_Last5Points"]:
        invalid = (~data[col].between(0, 15)).sum()
        print(f"Invalid {col}:", invalid)

    # Check H2H counts are consistent
    h2h_total = (
        data["H2H_HomeWins"]
        + data["H2H_AwayWins"]
        + data["H2H_Draws"]
    )
    print("Invalid H2H totals:",
          (h2h_total != data["H2H_Matches"]).sum())

    # Display remaining missing columns
    missing = data.isna().sum()
    print("\nRemaining missing values:")
    print(missing[missing > 0].to_string())


Bundesliga 1
Dataset shape: (2124, 67)
Duplicate matches: 0
Match outcomes: {'H': 933, 'A': 671, 'D': 520}
Invalid outcomes: 0
Missing rolling values: 0
Invalid Home_Last5Points: 0
Invalid Away_Last5Points: 0
Invalid H2H totals: 0

Remaining missing values:
Home_Prev5SeasonAvgPPG       261
Home_Prev5SeasonAvgPoints    261
Away_Prev5SeasonAvgPPG       261
Away_Prev5SeasonAvgPoints    261

Bundesliga 2
Dataset shape: (2142, 67)
Duplicate matches: 0
Match outcomes: {'H': 932, 'A': 641, 'D': 569}
Invalid outcomes: 0
Missing rolling values: 0
Invalid Home_Last5Points: 0
Invalid Away_Last5Points: 0
Invalid H2H totals: 0

Remaining missing values:
Home_Prev5SeasonAvgPPG       408
Home_Prev5SeasonAvgPoints    408
Away_Prev5SeasonAvgPPG       410
Away_Prev5SeasonAvgPoints    410


In [25]:
# Summarize all completed feature engineering for both Bundesliga divisions.
# Display dataset sizes, engineered feature categories, and example matches.
# This provides a clear overview of the final datasets for teammates.

feature_groups = {
    "Previous 5-Game Performance": [
        col for col in final_d1.columns
        if col.startswith(("Home_Last5", "Away_Last5"))
    ],
    "Previous 5-Season Performance": [
        col for col in final_d1.columns
        if col.startswith(("Home_Prev5", "Away_Prev5",
                           "Home_PrevSeasons", "Away_PrevSeasons"))
    ],
    "Head-to-Head Records": [
        col for col in final_d1.columns if col.startswith("H2H_")
    ],
    "Rest Days": ["Home_RestDays", "Away_RestDays"]
}

print("FINAL FEATURE ENGINEERING SUMMARY")
print("=" * 50)

print(f"Bundesliga 1: {final_d1.shape[0]} matches, {final_d1.shape[1]} columns")
print(f"Bundesliga 2: {final_d2.shape[0]} matches, {final_d2.shape[1]} columns")

print("\nENGINEERED FEATURES")

for category, features in feature_groups.items():
    print(f"\n{category} ({len(features)} features)")
    for feature in features:
        print(" -", feature)

# Display example matches with important engineered features
sample_cols = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "Home_Last5Points", "Away_Last5Points",
    "Home_Last5AvgGoalsFor", "Away_Last5AvgGoalsFor",
    "Home_Prev5SeasonAvgPPG", "Away_Prev5SeasonAvgPPG",
    "H2H_HomeWins", "H2H_AwayWins", "H2H_Draws",
    "Home_RestDays", "Away_RestDays", "FTR"
]

print("\nSAMPLE BUNDESLIGA 1 MATCHES")
display(final_d1[sample_cols].tail(5))

print("\nSAMPLE BUNDESLIGA 2 MATCHES")
display(final_d2[sample_cols].tail(5))

FINAL FEATURE ENGINEERING SUMMARY
Bundesliga 1: 2124 matches, 67 columns
Bundesliga 2: 2142 matches, 67 columns

ENGINEERED FEATURES

Previous 5-Game Performance (30 features)
 - Home_Last5Points
 - Home_Last5AvgGoalsFor
 - Home_Last5AvgGoalsAgainst
 - Home_Last5AvgShotsFor
 - Home_Last5AvgShotsAgainst
 - Home_Last5AvgShotsOnTargetFor
 - Home_Last5AvgShotsOnTargetAgainst
 - Home_Last5AvgCornersFor
 - Home_Last5AvgCornersAgainst
 - Home_Last5AvgFoulsFor
 - Home_Last5AvgFoulsAgainst
 - Home_Last5AvgYellowCardsFor
 - Home_Last5AvgYellowCardsAgainst
 - Home_Last5AvgRedCardsFor
 - Home_Last5AvgRedCardsAgainst
 - Away_Last5Points
 - Away_Last5AvgGoalsFor
 - Away_Last5AvgGoalsAgainst
 - Away_Last5AvgShotsFor
 - Away_Last5AvgShotsAgainst
 - Away_Last5AvgShotsOnTargetFor
 - Away_Last5AvgShotsOnTargetAgainst
 - Away_Last5AvgCornersFor
 - Away_Last5AvgCornersAgainst
 - Away_Last5AvgFoulsFor
 - Away_Last5AvgFoulsAgainst
 - Away_Last5AvgYellowCardsFor
 - Away_Last5AvgYellowCardsAgainst
 - Away_Last

,Season,Date,HomeTeam,AwayTeam,Home_Last5Points,Away_Last5Points,Home_Last5AvgGoalsFor,Away_Last5AvgGoalsFor,Home_Prev5SeasonAvgPPG,Away_Prev5SeasonAvgPPG,H2H_HomeWins,H2H_AwayWins,H2H_Draws,Home_RestDays,Away_RestDays,FTR
2119,26-27,2026-11-07,Stuttgart,Werder Bremen,6.0,11.0,1.6,2.4,1.476471,1.317647,1.0,2.0,2.0,7.0,7.0,H
2120,26-27,2026-11-07,Dortmund,Elversberg,9.0,7.0,1.2,1.6,1.958824,1.598039,0.0,0.0,0.0,7.0,8.0,H
2121,26-27,2026-11-08,Hoffenheim,Schalke 04,6.0,2.0,1.6,0.2,1.300000,1.452941,3.0,1.0,1.0,8.0,8.0,H
2122,26-27,2026-11-08,FC Koln,Leverkusen,5.0,9.0,0.8,2.0,1.258824,1.952941,1.0,4.0,0.0,7.0,8.0,A
2123,26-27,2026-11-08,Freiburg,Union Berlin,7.0,6.0,1.6,0.6,1.517647,1.358824,0.0,3.0,2.0,8.0,7.0,H



SAMPLE BUNDESLIGA 2 MATCHES


,Season,Date,HomeTeam,AwayTeam,Home_Last5Points,Away_Last5Points,Home_Last5AvgGoalsFor,Away_Last5AvgGoalsFor,Home_Prev5SeasonAvgPPG,Away_Prev5SeasonAvgPPG,H2H_HomeWins,H2H_AwayWins,H2H_Draws,Home_RestDays,Away_RestDays,FTR
2137,26-27,2026-11-07,Osnabruck,Kaiserslautern,5.0,10.0,1.2,0.8,0.990196,1.389706,0.0,1.0,1.0,7.0,7.0,A
2138,26-27,2026-11-07,Hertha,Wolfsburg,13.0,11.0,1.6,3.0,1.205882,1.176471,1.0,3.0,1.0,7.0,7.0,H
2139,26-27,2026-11-08,Bochum,Karlsruhe,4.0,6.0,0.8,1.4,1.052941,1.400000,2.0,1.0,2.0,9.0,8.0,H
2140,26-27,2026-11-08,Heidenheim,Hannover,3.0,7.0,1.0,1.0,1.270588,1.464706,4.0,1.0,0.0,8.0,7.0,D
2141,26-27,2026-11-08,Magdeburg,St Pauli,8.0,6.0,2.2,1.0,1.200000,1.423529,1.0,3.0,1.0,8.0,7.0,H


In [26]:
# Compare the number of matches in each season across both divisions.
# Identify which season explains the 18-match difference.

season_counts = (
    df.groupby(["Season", "Division"])
      .size()
      .unstack()
)

season_counts["Difference_D2_minus_D1"] = (
    season_counts["D2"] - season_counts["D1"]
)

display(season_counts)

Division,D1,D2,Difference_D2_minus_D1
Season,,,
18-19,306,306,0
19-20,306,306,0
20-21,306,306,0
21-22,306,306,0
22-23,306,306,0
23-24,306,306,0
24-25,306,306,0
25-26,306,306,0
26-27,81,99,18
